## Load packages

In [1]:
import sys, traceback
from plantcv import plantcv as pcv
import cv2
from cv2 import aruco

import numpy as np
import glob
import os
import pandas as pd
import timeit
import datetime # To get current time for saving CSV file

print("Packages loaded successfully!")

Packages loaded successfully!


### User-defined function to extract Aruco ID from image

In [2]:

def FindAruco(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    aruco_dict = aruco.Dictionary_get(aruco.DICT_5X5_250)
    parameters =  aruco.DetectorParameters_create()
    corners, ids, rejectedImgPoints = aruco.detectMarkers(gray, aruco_dict, parameters=parameters)

    return ids

## Inputs chunk

Just change the directories below:

In [5]:
inputdir = "C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/Ex8_Phenocage_July2023/071123"
outdir = "C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/Ex8_Phenocage_July2023/071123/Processed/"
resultsdir = "C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/Ex8_Phenocage_July2023/071123/Results/"

foldername = os.path.basename(os.path.normpath(resultsdir))

## Processing chunk

In [6]:
pcv.params.debug = "None"
# print(param_var)

os.getcwd()
os.chdir(inputdir)
image_list = glob.glob('*.jpg')
print("Total number of images = ", len(image_list))

start = timeit.default_timer()  
# =======================================
# =======================================
# =======================================

# PART 1 - Finding unique pots in the folder based on image name - e.g., "2022.12.16-11.57.20.RaspiZ"
pot_names = []
for i in range(len(image_list)):
    my_picture = image_list[i]
    pot_names.append(my_picture[0:26]) # Appending just the pot name ---> upto RaspiZ

    # get unique strings in the list
    unique_pots = list(set(pot_names)) # 'set' identifies the unique set of strings in the list and eliminates the duplicates
    # print(unique_pots)

print(f"Unique pots in the inputs folder = {len(unique_pots)}")

# =======================================
# =======================================
# =======================================

# PART 2 - Create a table that has "Unique pot" and its  respective "Aruco ID"
pot_aruco_df = pd.DataFrame(columns = ['Unique_image_name', 'Pot_number'])
# Now picking each unique pot and assigning a pot number to it by scanning the aruco marker from any of the views
for j in range(len(unique_pots)):        
    unique_pot_name = unique_pots[j]
    # print(unique_pot_name)

    # loop through each image between side1 to side7
    for s in range(7):
        side_img = f"{unique_pot_name}_side{s+1}.jpg"
        # print(side_img)

        img, path, img_filename = pcv.readimage(filename=side_img, mode="native")

        pot_aruco_id = FindAruco(img)

        if (pot_aruco_id != None):
            row_df = pd.DataFrame({'Unique_image_name' : [unique_pot_name], 'Pot_number' : [pot_aruco_id[0,0]]})
            pot_aruco_df = pd.concat([pot_aruco_df, row_df])
            break
        
# pot_aruco_df  # This will output all the unique pot names and its corresponding Aruco ID to the pot

end = timeit.default_timer()  

print(f"Time taken to assign ID numbers = {end-start: .2f} sec")

# =======================================
# =======================================
# =======================================

# PART 3 - Analyze images and get area, height, and width

# Create an empty dataframe to store area, height, width, and pot_id values from images
results_df = pd.DataFrame(columns = ['Image_name', 'Area'])
# print(results_df)

start = timeit.default_timer()

for k in range(len(image_list)):
    my_picture = image_list[k]
    pot_group_search = my_picture[0:26]    

    pcv.params.debug = "None"
    # STEP 1: Read image
    img, path, img_filename = pcv.readimage(filename=my_picture, mode="native")

    #=============
    # Getting pot ID from the previous table with all the unique pots and pot IDs
    get_pot_num = pot_aruco_df[pot_aruco_df['Unique_image_name'].apply(lambda x: pot_group_search in x)]
    pot_id = get_pot_num.Pot_number[0] # Extracting only the first element (0 index) from column Pot_number
    
    print(f"Processing...[{k+1}/{len(image_list)}] ---> {my_picture} ---> Pot #: {pot_id}")
    #=============
    
    # Corrects image based on color standard and stores output as corrected_img
    corrected_img = pcv.white_balance(img=img, mode='hist', roi=[1500, 1300, 100, 100])

    #=============
    # STEP 2: Convert image from RGB colorspace to LAB and HSV colorspace
    v = pcv.rgb2gray_hsv(rgb_img=corrected_img, channel='v')
    a = pcv.rgb2gray_lab(rgb_img=corrected_img, channel='a')

    #=============
    # STEP 3 Set a binary threshold on the saturation channel image
    # Threshold the a channel image
    v_thresh = pcv.threshold.binary(gray_img=v, threshold=110, max_value=111,object_type='dark')
    a_thresh1 = pcv.threshold.binary(gray_img=a, threshold=106, max_value=np.max(a), object_type='dark')

    # median filter on a_thresh image

    a_thresh = pcv.median_blur(a_thresh1, 2)

    pcv.params.debug = "None"
    #=============
    # STEP 4: Join the threshold from two layers of masking
    v_a = pcv.logical_or(bin_img1=v_thresh, bin_img2=a_thresh)

    #=============
    # STEP 5: Apply image mask based v_a combined channel
    masked = pcv.apply_mask(img=corrected_img , mask=v_a, mask_color='white')

    #=============
    # STEP 6 Identify objects
    # img - RGB or grayscale image data for plotting
    # mask - Binary mask used for detecting contours
    id_objects, obj_hierarchy = pcv.find_objects(img=masked, mask=a_thresh)

    #=============
    # STEP 7: Define region of interest (ROI)
    roi1, roi_hierarchy= pcv.roi.rectangle(img=masked, x=500, y=250, h=1060, w=1100)
    #=============
    # STEP 8: Keep objects that overlap with the ROI
    roi_objects, hierarchy3, kept_mask, obj_area = pcv.roi_objects(img=corrected_img, roi_contour=roi1,
                                                                roi_hierarchy=roi_hierarchy,
                                                                object_contour=id_objects,
                                                                obj_hierarchy=obj_hierarchy,
                                                                roi_type='partial')

    #=============
    # STEP 9 Object combine kept objects
    # Inputs:
    #   img - RGB or grayscale image data for plotting
    #   contours - Contour list
    #   hierarchy - Contour hierarchy array
    obj, mask = pcv.object_composition(img=corrected_img, contours=roi_objects, hierarchy=hierarchy3)

    img_copy = np.copy(corrected_img)

    #=============
    # Filter objects by ROI
    roi = roi1
    hierarchy = roi_hierarchy

    filtered_contours, filtered_hierarchy, filtered_mask, filtered_area = pcv.roi_objects(
        img=img_copy, roi_type="partial", roi_contour=roi,roi_hierarchy = hierarchy,object_contour=roi_objects, 
        obj_hierarchy=hierarchy3)

    # Combine objects together in each plant
    plant_contour, plant_mask = pcv.object_composition(img=img_copy, contours=filtered_contours, hierarchy=filtered_hierarchy)

    if(len(filtered_contours) == 0):
        # Something
        print("No plant!")
        img_copy =  img
        plant_area = 0
    else: 

        # Analyze the shape of each plant
        analysis_images = pcv.analyze_object(img=img_copy, obj=plant_contour, mask=plant_mask)
        analysis_images

        # Save the image with shape characteristics
        img_copy = analysis_images
        
        pcv.params.debug = "None"

        # Look at object area data without writing to a file 
        plant_area = pcv.outputs.observations['default']['area']['value']
        plant_height = pcv.outputs.observations['default']['height']['value']
        plant_width = pcv.outputs.observations['default']['width']['value']
        # print(f"Area = {plant_area}; height = {plant_height}; width = {plant_width}")

    outfile = os.path.join(outdir, my_picture)
    pcv.print_image(img=img_copy, filename=outfile)

    # Create the new row as its own dataframe
    row_df = pd.DataFrame({'Image_name' : [my_picture], 'Area' : [plant_area], 'Height' : [plant_height], 'Width' : [plant_width], 'Pot_no' : [pot_id]})
    results_df = pd.concat([results_df, row_df])

end = timeit.default_timer()

print(f"/n Time taken to process all images = {end-start: .2f} sec")

Total number of images =  1001
Unique pots in the inputs folder = 143
Time taken to assign ID numbers =  32.43 sec
Processing...[1/1001] ---> 2023.07.11-13.43.47.RaspiZ_side1.jpg ---> Pot #: 1
Processing...[2/1001] ---> 2023.07.11-13.43.47.RaspiZ_side2.jpg ---> Pot #: 1
Processing...[3/1001] ---> 2023.07.11-13.43.47.RaspiZ_side3.jpg ---> Pot #: 1
Processing...[4/1001] ---> 2023.07.11-13.43.47.RaspiZ_side4.jpg ---> Pot #: 1
Processing...[5/1001] ---> 2023.07.11-13.43.47.RaspiZ_side5.jpg ---> Pot #: 1
Processing...[6/1001] ---> 2023.07.11-13.43.47.RaspiZ_side6.jpg ---> Pot #: 1
Processing...[7/1001] ---> 2023.07.11-13.43.47.RaspiZ_side7.jpg ---> Pot #: 1
Processing...[8/1001] ---> 2023.07.11-13.44.55.RaspiZ_side1.jpg ---> Pot #: 2
Processing...[9/1001] ---> 2023.07.11-13.44.55.RaspiZ_side2.jpg ---> Pot #: 2
Processing...[10/1001] ---> 2023.07.11-13.44.55.RaspiZ_side3.jpg ---> Pot #: 2
Processing...[11/1001] ---> 2023.07.11-13.44.55.RaspiZ_side4.jpg ---> Pot #: 2
Processing...[12/1001] --

### 2. Save results to folder

In [7]:
# get the current time and date and format it
now = datetime.datetime.now()
formatted_date = now.strftime("%Y.%m.%d_%H.%M.%S")

csv_name = f"Results_{formatted_date}.csv"

results_df.to_csv(f"{resultsdir}{csv_name}", index=False)
print("Results successfully saved in * " + str(foldername) + " * folder!")


Results successfully saved in * Results * folder!
